# 🚀 AIC 2026 — End-to-End Pipeline


### Pipeline 6 bước
1. Giải nén dữ liệu BTC
2. Tạo Metadata & Trích xuất Transcript âm thanh (Whisper)
3. **Huấn luyện LoRA CLIP** (GPU AMD — batch=4, accum=8)
4. **Trích xuất Feature Vectors** (GPU AMD CUDA — inference siêu nhanh)
5. Đẩy Vectors lên Qdrant Vector Database
6. Thử nghiệm tìm kiếm

## 📦 Bước 0: Cài đặt thư viện

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Kiểm tra GPU AMD DirectML
import torch
import torch_directml

print(f"PyTorch: {torch.__version__}")
print(f"DirectML available: {torch_directml.is_available()}")
print(f"GPU device: {torch_directml.device()}")
print(f"GPU name: {torch_directml.device_name(0)}")

## 📂 Bước 1: Giải nén dữ liệu BTC
Tự động giải nén các file `.zip` trong `ZIP/` vào đúng cấu trúc `data/`.

In [ ]:
# Xem trước các file sẽ giải nén
!python scripts/extract_btc_data.py --dry-run

In [ ]:
# Giải nén thực tế
!python scripts/extract_btc_data.py

## 📄 Bước 2: Import Metadata & Transcript
Quét keyframes, map frame_id, tích hợp lời thoại âm thanh từ Whisper vào `data/index/metadata.jsonl`.

In [ ]:
!python scripts/import_btc_data.py --with-transcript

## 🧠 Bước 1: Huấn luyện LoRA Fine-Tuning cho CLIP
Fine-tune CLIP ViT-B/32 bằng kỹ thuật LoRA (chỉ cập nhật 0.3% tham số - 1.7MB).


### 🧪 Chạy Thử (Test) - 100 Keyframes
Dùng để kiểm tra xem code có chạy lỗi hay không. Sẽ hoàn thành trong chưa tới 1 phút.


In [ ]:
# Test trên GPU NVIDIA (CUDA)
!python scripts/train_lora_clip.py --epochs 1 --batch-size 32 --limit 100

### 🚀 Chạy Thật (Main) - Toàn bộ dữ liệu
Dùng để huấn luyện toàn bộ 44.496 ảnh. Vì chạy bằng GPU, quá trình này chỉ tốn khoảng 20-30 phút.

In [ ]:
# Chạy thật trên GPU NVIDIA (CUDA)
!python scripts/train_lora_clip.py --epochs 3 --batch-size 32

## 🖼️ Bước 2: Trích xuất Vector Đặc Trưng (Features Extraction)
Sử dụng mô hình CLIP (đã gắn LoRA) để mã hoá toàn bộ ảnh thành vector 512 chiều.


In [ ]:
# Cài đặt và nạp mô hình vào GPU (hoặc CPU nếu GPU lỗi)
from pipeline_batch_run import step2_extract_features
from backend.embedding.clip_encoder import _load_clip
from backend.config import DEVICE
import torch

# Nạp mô hình đã có LoRA (Tự động tải lora_weights.pt)
model, preprocess = _load_clip()
model = model.to(DEVICE)

### 🧪 Chạy Thử (Test) - 100 Keyframes
Dùng để kiểm tra luồng trích xuất.


In [ ]:
step2_extract_features(model, preprocess, device=DEVICE, limit=100)

### 🚀 Chạy Thật (Main) - Toàn bộ dữ liệu
Sẽ mất khoảng 3-5 phút nếu dùng GPU, hoặc 1-2 tiếng nếu dùng CPU.


In [ ]:
step2_extract_features(model, preprocess, device=DEVICE, limit=0)

## ⚡ Bước 3: Đóng gói FAISS Index
Gom toàn bộ các file .npy rải rác lại thành một cơ sở dữ liệu Vector (FAISS Index) để tìm kiếm siêu tốc.


### 🧪 Chạy Thử (Test) - 100 Keyframes


In [ ]:
from pipeline_batch_run import step3_build_faiss_index
index, metadata = step3_build_faiss_index(limit=100)

### 🚀 Chạy Thật (Main) - Toàn bộ dữ liệu
Chạy đa luồng (32 workers), mất chưa tới 5 giây.


In [ ]:
from pipeline_batch_run import step3_build_faiss_index
index, metadata = step3_build_faiss_index(limit=0)

## 🔍 Bước 4: Thử nghiệm Tìm Kiếm (Search & Visualize)
Truy vấn văn bản và xem kết quả.


In [ ]:
from backend.embedding.clip_encoder import encode_text_raw
import numpy as np

query = "a photo of a tree"
vec = encode_text_raw(query)
print(f"Query: '{query}' -> Vector shape: {vec.shape}, norm: {np.linalg.norm(vec):.4f}")

In [ ]:
import faiss
import numpy as np

query_super = vec.reshape(1, -1).astype(np.float32)
faiss.normalize_L2(query_super)

top_k = 10
scores, indices = index.search(query_super, top_k)

print(f"\n🔍 Top 10 kết quả cho: '{query}'\n")
for i in range(top_k):
    idx = indices[0][i]
    score = scores[0][i]
    if idx < 0 or idx >= len(metadata): continue
    p = metadata[idx]
    print(f"  #{i+1:02d} | Score: {score:.4f} | {p.get('video_id','')} | Frame: {p.get('frame_id','')} | {p.get('pts_time',0):.1f}s")

In [ ]:
# Hiển thị ảnh keyframe từ kết quả tìm kiếm
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

n = min(top_k, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f"Kết quả tìm kiếm: '{query}'", fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < n:
        idx = indices[0][i]
        score = scores[0][i]
        
        if idx < 0 or idx >= len(metadata):
            ax.axis('off')
            continue
            
        p = metadata[idx]
        img_path = Path(p.get('path', ''))
        vid = p.get('video_id', '')
        fid = p.get('frame_id', 0)
        pts = p.get('pts_time', 0.0)

        if img_path.exists():
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'#{i+1} | Score: {score:.4f}\n{vid} | Frame {fid} | {pts:.1f}s', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()